# SOLA with Sobolev Model Space and Straight-Ray Data

This notebook demonstrates SOLA property estimation in a 3D box domain using:

- **Sobolev H² model space** built from a `BesselSobolevBox` weight operator
- **Straight-ray path-integral measurements** (`PathIntegralFunctional`)
- **Gaussian volume-kernel target forms** (`VolumeKernel3D`)

The SOLA matrix is assembled manually from Gram matrices, enabling direct
inspection of the representer structure and unimodularity correction.

**Outline**

1. Sobolev model space construction  
2. Ray geometry (10 stations at z=1, 30 sources in volume)  
3. Forward forms and true model  
4. SOLA matrix construction (representers → Gram matrices → X)  
5. Property inference (mean, uncertainty, table)  
6. Unimodularity constraint (rank-1 correction)  
7. Resolving kernels vs target Gaussians

In [ ]:
import sys
import time
from pathlib import Path

# Bootstrap sys.path — works regardless of Jupyter kernel CWD
def _find_workspace_root(start: Path) -> Path:
    """Walk upward until we find the directory containing pygeoinf and pygeoinf3D."""
    p = start.resolve()
    for _ in range(8):
        if (p / "pygeoinf").is_dir() and (p / "pygeoinf3D").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(f"Cannot find workspace root from {start}")

WORKSPACE_ROOT = _find_workspace_root(Path.cwd())
for repo_root in [WORKSPACE_ROOT / "pygeoinf", WORKSPACE_ROOT / "pygeoinf3D"]:
    repo_root_str = str(repo_root)
    if repo_root.exists() and repo_root_str not in sys.path:
        sys.path.insert(0, repo_root_str)

import numpy as np
from scipy.interpolate import RegularGridInterpolator
import plotly.graph_objects as go

from pygeoinf import EuclideanSpace, CholeskySolver, MatrixLinearOperator
from pygeoinf3d import BoxDomain, Lebesgue3D, Function3D
from pygeoinf3d.providers.tensor_product import BoxLaplacianSpectrumProvider
from pygeoinf3d.operators.bessel_sobolev_box import BesselSobolevBox
from pygeoinf3d.operators.sola import SOLAOperator3D
from pygeoinf3d.spaces.sobolev import Sobolev3D
from pygeoinf3d.spaces.forms import PathIntegralFunctional, VolumeKernel3D
from pygeoinf3d.core.paths import Path3D
from pygeoinf3d.providers.function_providers import GaussianProvider3D
from pygeoinf3d.plot import (
    plot_function3d_axis_slider,
    compare_function3d_axis_sliders,
    compare_panels_axis_dropdown_sliders,
)

print("Imports OK")
print(f"Workspace root: {WORKSPACE_ROOT}")


## 1. Sobolev Model Space

We build a **Sobolev H²** space on $[0,1]^3$ using the Bessel–Sobolev weight

$$\langle f, g \rangle_{H^2} = \langle W f,\, g \rangle_{L^2}, \quad W = (I - \Delta)^2$$

implemented spectrally via `BesselSobolevBox` with `shift=1.0`, `order=2.0` and Neumann eigenfunctions truncated at $(8,8,8)$.

In [ ]:
np.random.seed(42)

box = BoxDomain(np.zeros(3), np.ones(3))
base_space = Lebesgue3D(box)

spec_provider = BoxLaplacianSpectrumProvider(box, truncation=(8, 8, 8), bc_type="neumann")
W     = BesselSobolevBox(base_space, spec_provider, shift=1.0, order=2.0)
W_inv = BesselSobolevBox(base_space, spec_provider, shift=1.0, order=-2.0)
M = Sobolev3D(base_space, W, W_inv)
n_modes = 8 ** 3
print(f"Box              : [0,1]^3")
print(f"Spectral modes   : (8,8,8)  →  {n_modes} modes")
print(f"Weight operator  : BesselSobolevBox  shift=1.0, order=2.0  (H²)")
print(f"Sobolev3D ready  : {M}")

## 2. Ray Geometry

**Setup**: 10 receivers fixed at $z=1$ (surface), 30 sources randomly placed
in $[0.05, 0.95]^3$.  Each source is connected to one randomly assigned station
by a straight ray.

In [ ]:
N_d = 100
N_stations = 10
np.random.seed(42)

# Stations: random (x,y) in [0.05,0.95], fixed z=1 (surface)
station_xy = np.random.uniform(0.05, 0.95, size=(N_stations, 2))
stations = np.column_stack([station_xy, np.ones(N_stations)])

# Sources: random in [0.05,0.95]^3
sources = np.random.uniform(0.05, 0.95, size=(N_d, 3))

# Assign each source to a (uniformly) random station
station_idx = np.random.randint(0, N_stations, size=N_d)

# Straight-ray Path3D: source → assigned station
paths = [
    Path3D.from_polyline(np.array([sources[i], stations[station_idx[i]]]))
    for i in range(N_d)
]
print(f"Stations : {N_stations} at z=1 (random x,y)")
print(f"Sources  : {N_d} in [0.05,0.95]^3")
print(f"Rays     : {len(paths)}")

# --- 3×3×3 target grid (for geometry preview) ---
_tgx = np.linspace(0.1, 0.9, 5)
target_grid = np.array([[x, y, z] for x in _tgx for y in _tgx for z in _tgx])
N_p = len(target_grid)

# --- Plotly 3D figure ---
fig_geo = go.Figure()
for i in range(N_d):
    src, sta = sources[i], stations[station_idx[i]]
    fig_geo.add_trace(go.Scatter3d(
        x=[src[0], sta[0]], y=[src[1], sta[1]], z=[src[2], sta[2]],
        mode="lines", line=dict(color="steelblue", width=2),
        showlegend=(i == 0), name="Rays" if i == 0 else None, legendgroup="rays",
    ))
fig_geo.add_trace(go.Scatter3d(
    x=stations[:, 0], y=stations[:, 1], z=stations[:, 2],
    mode="markers", marker=dict(color="red", size=7, symbol="square"), name="Stations",
))
fig_geo.add_trace(go.Scatter3d(
    x=sources[:, 0], y=sources[:, 1], z=sources[:, 2],
    mode="markers", marker=dict(color="limegreen", size=4, symbol="circle"), name="Sources",
))
fig_geo.add_trace(go.Scatter3d(
    x=target_grid[:, 0], y=target_grid[:, 1], z=target_grid[:, 2],
    mode="markers", marker=dict(color="orange", size=4, symbol="diamond"), name="Target locations",
))
fig_geo.update_layout(
    title="Ray geometry: sources → stations, and SOLA target locations",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z",
               xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]), zaxis=dict(range=[0, 1])),
)
fig_geo.show()

## 3. Forward Forms and True Model

Each measurement is a path integral:

$$d_i = \int_{\gamma_i} m(\mathbf{x})\, \mathrm{d}s + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

The true model is a sum of three Gaussians with different amplitudes and widths.

In [ ]:
# Forward forms: PathIntegralFunctional on the Sobolev space
forms_G = [PathIntegralFunctional(M, p, n_quad=200) for p in paths]
print(f"Forward forms G: {len(forms_G)} PathIntegralFunctional on Sobolev3D")

D = EuclideanSpace(N_d)
P = EuclideanSpace(N_p)

G = SOLAOperator3D(M, D, forms_G)
# True model: three Gaussians
def _true_model_callable(x):
    x = np.asarray(x, dtype=float)
    single = x.ndim == 1
    pts = x[None, :] if single else x
    c1 = np.array([0.3, 0.5, 0.5])
    c2 = np.array([0.7, 0.4, 0.5])
    c3 = np.array([0.5, 0.6, 0.3])
    val = (
        1.5 * np.exp(-np.sum((pts - c1) ** 2, axis=1) / (2 * 0.12 ** 2))
        + 1.0 * np.exp(-np.sum((pts - c2) ** 2, axis=1) / (2 * 0.10 ** 2))
        - 0.8 * np.exp(-np.sum((pts - c3) ** 2, axis=1) / (2 * 0.08 ** 2))
    )
    return float(val[0]) if single else val

m_true = Function3D(box, evaluate_callable=_true_model_callable, name="true model")

# Synthetic data with 5% noise
d_bar = G(m_true)
noise_std = 0.05 * np.max(np.abs(d_bar))
C_D_mat = noise_std ** 2 * np.eye(N_d)
C_D = MatrixLinearOperator(D, D, C_D_mat)
rng = np.random.default_rng(123)
d_tilde = d_bar + rng.normal(0, noise_std, size=N_d)

snr = np.linalg.norm(d_bar) / np.linalg.norm(d_tilde - d_bar)
print(f"Noise std : {noise_std:.4e}")
print(f"SNR       : {snr:.1f}")

# Target forms: 3x3x3 Gaussian volume kernels on base_space (L2)
N_p = len(target_grid)  # 27
fp_T = GaussianProvider3D(box, centers=target_grid, sigmas=0.10, normalized=True)
forms_T = [VolumeKernel3D(M, fp_T.get_function(k)) for k in range(N_p)]
print(f"Target forms T : {N_p} VolumeKernel3D on {_tgx.shape[0]}x{_tgx.shape[0]}x{_tgx.shape[0]} grid")

T = SOLAOperator3D(M, P, forms_T)

# Visualise true model
fig_true = plot_function3d_axis_slider(
    m_true, axis="z", n_slices=10, resolution=60,
    title="True model  (z-axis slider)", show_plot=False,
)
fig_true.show()


In [ ]:
t0 = time.time()

# — Step 1: Sobolev representers  r_i = W⁻¹ℓᵢ* —
print(f"Computing {N_d} Sobolev representers (spectral, 8³=512 modes per representer)...")
representers = []
for i, f in enumerate(forms_G):
    representers.append(M.from_dual(f))
    if (i + 1) % 10 == 0:
        print(f"  representer {i + 1}/{N_d}  [{time.time() - t0:.1f}s]")
print(f"All {N_d} representers done.  [{time.time() - t0:.1f}s]")

solver = CholeskySolver(galerkin=True, parallel=True, n_jobs=-1)
X = T @ G.adjoint @ solver(G @ G.adjoint + C_D)

print(f"Total SOLA construction time   : {time.time() - t0:.1f}s")


## 4b. Riesz Representers of Path Forms

The **Riesz representer** $r_i \in H^2$ of the $i$-th path-integral form $\ell_i$ satisfies

$$\langle r_i,\, f \rangle_{H^2} = \ell_i(f) \quad \forall f \in H^2$$

computed as $r_i = W^{-1}\ell_i^*$. Its shape reflects the ray geometry smoothed by the $H^2$ norm — elongated along the ray and attenuated where the path does not pass. The dropdown below selects among 5 representative rays.

In [ ]:
# 5 representative ray indices (spread across the 30 rays)
REP_IDX = [0, 6, 14, 21, 28]

rep_fns  = [representers[i] for i in REP_IDX]

def _ray_label(i):
    src = sources[i]
    sta = stations[station_idx[i]]
    return (f"Ray {i}  src=({src[0]:.2f},{src[1]:.2f},{src[2]:.2f})"
            f" → sta=({sta[0]:.2f},{sta[1]:.2f},{sta[2]:.2f})")

rep_labels = [_ray_label(i) for i in REP_IDX]

fig_rep = compare_panels_axis_dropdown_sliders(
    panel_functions=[rep_fns],
    panel_labels=[rep_labels],
    panel_titles=["Riesz representer  $r_i = W^{-1}\\ell_i^*$"],
    axis="z",
    n_slices=12,
    resolution=60,
    show_plot=False,
)
fig_rep.show()


## 4c. Total Sensitivity

The **total sensitivity** (also called the *data kernel sum*) is

$$S(x) = \sum_{i=1}^{N_d} r_i(x)$$

It shows which parts of the domain are collectively well-sampled by all rays: high values indicate regions that strongly influence the data, while low values mark blind spots. The z-slider below shows horizontal slices through $S$.

In [ ]:
total_sensitivity = G.adjoint(np.ones(N_d))

fig_sens = plot_function3d_axis_slider(
    total_sensitivity,
    axis="z",
    n_slices=15,
    resolution=70,
    title="Total sensitivity  S(x) = Σ rᵢ(x)  (z-slider)",
    show_plot=False,
)
fig_sens.show()

## 5. Property Inference

Apply the SOLA matrix to the noisy data to estimate the 27 target averaged properties:

In [ ]:
p_mean = X(d_tilde)
p_cov  = X @ C_D @ X.adjoint
p_cov_mat = p_cov.matrix(dense=True, galerkin=True, parallel=True, n_jobs=-1)
p_std  = np.sqrt(np.diag(p_cov_mat))
p_true = np.array([forms_T[k](m_true) for k in range(N_p)])

n_print = min(20, N_p)
print(f"(showing first {n_print} of {N_p} properties)")
print("  k  |  (cx,  cy,  cz)       |   true    |   mean    |   std    ")
print("  " + "-" * 62)
for k in range(n_print):
    cx, cy, cz = target_grid[k]
    print(
        f"  {k:2d} | ({cx:.2f}, {cy:.2f}, {cz:.2f})"
        f" |  {p_true[k]:8.4f} |  {p_mean[k]:8.4f} |  {p_std[k]:8.4f}"
    )

# --- Build Function3D fields via RegularGridInterpolator ---
_grid_ax = np.unique(target_grid[:, 0])   # [0.1, 0.5, 0.9]
_n_ax = len(_grid_ax)

def _make_prop_fn(values, name=""):
    grid_vals = values.reshape(_n_ax, _n_ax, _n_ax)
    _lo, _hi = _grid_ax[0], _grid_ax[-1]
    _interp = RegularGridInterpolator(
        (_grid_ax, _grid_ax, _grid_ax), grid_vals,
        method="linear", bounds_error=False, fill_value=0.0,
    )
    def _fn(x):
        pts_c = np.clip(np.atleast_2d(x), _lo, _hi)
        return _interp(pts_c).squeeze()
    return Function3D(box, evaluate_callable=_fn, name=name)

p_mean_field = _make_prop_fn(p_mean, "SOLA mean")
p_2std_field = _make_prop_fn(2 * p_std, "Uncertainty 2σ")

fig_prop = compare_function3d_axis_sliders(
    [p_mean_field, p_2std_field],
    axis="z", n_slices=10, resolution=60,
    titles=["SOLA estimate (mean)", "Uncertainty (2σ)"],
    show_plot=False, width=1000, height=500,
)
fig_prop.show()

## 6. Unimodularity

A SOLA estimator is **unimodular** if $X \mathbf{G}\mathbf{1} = \mathbf{1}$, i.e. the estimator
exactly recovers a spatially constant model with value 1.  
We impose this via a minimal rank-1 correction:

$$X_{\mathrm{uni}} = X + \boldsymbol{\alpha}\, (G\mathbf{1})^\top, \quad
\alpha_k = \frac{1 - (X G\mathbf{1})_k}{\|G\mathbf{1}\|^2}$$

In [ ]:
ones_fn = Function3D(
    box,
    evaluate_callable=lambda x: np.ones(np.atleast_2d(x).shape[0]),
    name="ones",
)
v = np.array([f(ones_fn) for f in forms_G])   # G(1) — forward forms applied to ones
w = np.ones(N_p)                               # desired property values of ones
# Rank-1 correction using the data-space metric (C_GG + C_D).
# Minimises the change to X while enforcing X_uni @ v = w.
u = np.linalg.solve(C_GG + C_D, v)            # (C_GG + C_D)^{-1} v
beta = float(np.dot(v, u))
X_uni = X - np.outer((X @ v - w) / beta, u)

check = X_uni @ v
print("X_uni(G 1) — should equal ones (first 5):", np.asarray(check).round(4)[:5])

# Unimodular mean and uncertainty
p_mean_uni = X_uni @ d_tilde
P_cov_uni  = X_uni @ C_D @ X_uni.T
p_std_uni  = np.sqrt(np.diag(P_cov_uni))

p_mean_uni_field = _make_prop_fn(p_mean_uni, "Unimodular mean")
p_2std_uni_field = _make_prop_fn(2 * p_std_uni, "Unimodular 2σ")

# Plot 1: unconstrained vs unimodular mean
fig_uni_mean = compare_function3d_axis_sliders(
    [p_mean_field, p_mean_uni_field], axis="z", n_slices=10, resolution=60,
    titles=["Mean — unconstrained X", "Mean — unimodular X"],
    show_plot=False, width=1000, height=500,
)
fig_uni_mean.show()

# Plot 2: unconstrained vs unimodular 2σ
fig_uni_unc = compare_function3d_axis_sliders(
    [p_2std_field, p_2std_uni_field], axis="z", n_slices=10, resolution=60,
    titles=["2σ — unconstrained X", "2σ — unimodular X"],
    show_plot=False, width=1000, height=500,
)
fig_uni_unc.show()

# Plot 3: unimodular mean | unimodular 2σ
fig_uni_est = compare_function3d_axis_sliders(
    [p_mean_uni_field, p_2std_uni_field], axis="z", n_slices=10, resolution=60,
    titles=["SOLA estimate (unimodular mean)", "Uncertainty (2σ) — unimodular"],
    show_plot=False, width=1000, height=500,
)
fig_uni_est.show()

## 7. Resolving Kernels

The **resolving kernel** for target $k$ is the function

$$A_k(\mathbf{x}) = \sum_{i=1}^{N_d} X_{\mathrm{uni}}[k,i]\, r_i(\mathbf{x})$$

It should be concentrated near the target Gaussian $g_k$.  
Below we compare resolving kernels to their corresponding target Gaussians for 5 selected targets.

In [ ]:
PREVIEW_IDX = [0, 6, 13, 20, 26]

rk_fns = [G.adjoint(X_uni[k, :]) for k in PREVIEW_IDX]
tgt_fns = [fp_T.get_function(k) for k in PREVIEW_IDX]

rk_labels = [
    f"Resolving k={k} ({target_grid[k, 0]:.2f},{target_grid[k, 1]:.2f},{target_grid[k, 2]:.2f})"
    for k in PREVIEW_IDX
]
tgt_labels = [
    f"Target k={k} ({target_grid[k, 0]:.2f},{target_grid[k, 1]:.2f},{target_grid[k, 2]:.2f})"
    for k in PREVIEW_IDX
]

fig_rk = compare_panels_axis_dropdown_sliders(
    panel_functions=[rk_fns, tgt_fns],
    panel_labels=[rk_labels, tgt_labels],
    panel_titles=["Resolving kernels", "Target Gaussians"],
    axis="z", n_slices=10, resolution=60,
    show_plot=False,
)
fig_rk.show()

## 8. Least-Norm Solution

In the Sobolev $H^2$ model space the minimum-norm data-fit estimate is
$$
    \hat{m} = \sum_{i=1}^{N_d} \alpha_i\, r_i, \qquad
    \boldsymbol{\alpha} = (C_{GG} + C_D)^{-1}\tilde{\mathbf{d}},
$$
where $r_i$ are the Sobolev representers already computed in §4.  
Below we compare $\hat{m}$ against the true model side-by-side.

In [ ]:
# Least-norm coefficients  alpha = (C_GG + C_D)^{-1} d_tilde
alpha = np.linalg.solve(C_GG + C_D, d_tilde)

# m_hat = G.adjoint(alpha)
# Adjoint chain: R_W (identity for EuclideanSpace), G* (SumLinearForm),
# sobolev.from_dual (spectral inversion = sum of representers)
m_hat = G.adjoint(alpha)   # Function3D in H2

# Verify: G(m_hat) should recover d_tilde (up to noise regularisation)
rel_res = np.linalg.norm(G(m_hat) - d_tilde) / np.linalg.norm(d_tilde)
print(f"Relative data residual  ||G m_hat - d_tilde|| / ||d_tilde|| : {rel_res:.4f}")

# Side-by-side comparison: true model vs least-norm solution
fig_lns = compare_function3d_axis_sliders(
    [m_true, m_hat], axis='z', n_slices=12, resolution=60,
    titles=["True model $m_\mathrm{true}$", "Least-norm solution $\hat{m}$"],
    show_plot=True, width=1000, height=500,
)